# KPCA-DA Application


In [ ]:
## PACKAGES
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
import umap.umap_ as umap
import torch
import torch.nn.functional as F
import torchvision.transforms as transforms
import timm
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
import seaborn as sn
import matplotlib.pyplot as plt

from IPython.display import display, clear_output
from pathlib import Path
import sys

data_root = Path("/Users/mymac/SHELIAPC/Zmyfiles/codes/research/KPCA-DA/KPCA-DA/examples/Data")

if not data_root.is_dir():
    raise FileNotFoundError(f"Data directory not found: {data_root}")

# all library code (kpcada.py, the baselines, pyDeepInsightv3) is the kpcada/ package in this notebook's folder
lib_dir = Path.cwd()
if not (lib_dir / "kpcada" / "__init__.py").is_file():
    raise FileNotFoundError(f"kpcada package not found in {lib_dir}; start the notebook from its own folder")
sys.path.insert(0, str(lib_dir))

#from kpcada.pyDeepInsightv3 import ImageTransformer



## Load data

In [ ]:
data_dir = data_root / "cptacvssmc"

def load_expr(path):
    """genes x samples CSV (first column = gene, last row = PAM50 label) -> samples x genes matrix, labels, gene names."""
    expr = pd.read_csv(path, encoding='utf-8', low_memory=False).transpose()
    expr.columns = expr.iloc[0]
    X = expr.iloc[1:, 0:-1].values.astype(float)
    Y = expr.iloc[1:, -1].values
    genes = expr.iloc[:, :-1].columns.to_numpy()
    return X, Y, genes

## Data1 (source): CPTAC
X1, Y1, genes1 = load_expr(data_dir / "smc_labels.csv")

## Data2 (target): SMC
X2, Y2, genes2 = load_expr(data_dir / "cptac_labels.csv")

print(f"source (CPTAC): {X1.shape[0]} samples x {X1.shape[1]} genes, classes {dict(zip(*np.unique(Y1, return_counts=True)))}")
print(f"target (SMC):   {X2.shape[0]} samples x {X2.shape[1]} genes, classes {dict(zip(*np.unique(Y2, return_counts=True)))}")

In [ ]:
# Split X2 dataset into 70-30 (70% for adaptation and 30% for testing)
X2_adapt, X2_test, Y2_adapt, Y2_test = train_test_split(
    X2, Y2, test_size=0.3, random_state=42, stratify=Y2)

In [ ]:
from sklearn.decomposition import PCA, KernelPCA

In [ ]:
#Preprocessing with variance thresholding and Label Encoder
# The variance filter is fitted once per domain: on the source (X1) and on the target ADAPTATION set (X2_adapt).
# The target mask is applied to both X2_adapt and X2_test, so the test samples keep exactly the genes of the
# adaptation-set layout. The filtered arrays (*_vf) and gene lists feed every KPCA, the KSP kernel and the images.

X_train, X_test, y_train, y_test = X1,X2_test,Y1,Y2_test

var_filter1 = X1.var(0) >= np.percentile(X1.var(0), 10., method='nearest')              # source genes
var_filter2 = X2_adapt.var(0) >= np.percentile(X2_adapt.var(0), 10., method='nearest')  # target genes (adaptation set)

X1_vf, genes1_vf = X1[:, var_filter1], genes1[var_filter1]
X2_adapt_vf, X2_test_vf, genes2_vf = X2_adapt[:, var_filter2], X2_test[:, var_filter2], genes2[var_filter2]
print(f"source: {len(genes1_vf)}/{len(genes1)} genes kept; target: {len(genes2_vf)}/{len(genes2)} genes kept")

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

le_mapping = dict(zip(le.transform(le.classes_), le.classes_))
num_classes = np.unique(y_train_enc).size

In [ ]:
from scipy.spatial.distance import cdist


def mmd2_rbf(A, B, gamma=None):
    if gamma is None:
        d2 = cdist(np.vstack([A, B]), np.vstack([A, B]), "sqeuclidean")
        gamma = 1.0 / np.median(d2[d2 > 0])
    k = lambda U, V: np.exp(-gamma * cdist(U, V, "sqeuclidean"))
    return k(A, A).mean() + k(B, B).mean() - 2 * k(A, B).mean()

In [ ]:
## Baseline: common genes -> KPCA fit on source, target projected -> MMD
from kpcada.kpca_procrustes import median_heuristic_gamma
from sklearn.decomposition import KernelPCA## MMD helper



common, i1, i2 = np.intersect1d(genes1_vf, genes2_vf, return_indices=True)
Xs_c = X1_vf[:, i1]       # source samples x common genes
Xt_c = X2_test_vf[:, i2]  # target test samples x common genes

# fit on source only; target never influences the axes
kpca = KernelPCA(n_components=2, kernel="rbf", gamma=median_heuristic_gamma(Xs_c))
Zs_common = kpca.fit_transform(Xs_c)
Zt_common = kpca.transform(Xt_c)


# --- same common-gene data, with Procrustes: genes as points, every common gene is an anchor ---
from kpcada import ProcrustesKPCA
kp_c = ProcrustesKPCA(n_components=2)
Zs_cp, Zt_cp = kp_c.fit_transform(Xs_c, Xt_c, common, common)   # (n_common, 2) each, Zt_cp rotated onto Zs_cp
print(f"MMD^2 (common genes, without Procrustes ): {mmd2_rbf(Zs_cp, kp_c.Z_target_):.6f}")
print(f"MMD^2 (common genes, with Procrustes ):   {mmd2_rbf(Zs_cp, Zt_cp):.6f}")


In [ ]:
## KPCA + Procrustes alignment (source is the reference), then MMD
from kpcada import ProcrustesKPCA

# source = X1_vf; target adaptation set = X2_adapt_vf, target test set = X2_test_vf (70-30 split in cell 4, variance-filtered in cell 6)
# each target set gets its own gene kPCA, then is rotated onto the same source embedding

# adapt -> source
kp_a = ProcrustesKPCA(n_components=2)
Zs_a, Za = kp_a.fit_transform(X1_vf, X2_adapt_vf, genes1_vf, genes2_vf)   # gene x 2 each (Za rotated onto Zs)

# test -> source
kp = ProcrustesKPCA(n_components=2)
Zs, Zt = kp.fit_transform(X1_vf, X2_test_vf, genes1_vf, genes2_vf)        # gene x 2 each (Zt rotated onto Zs)
Zt_unaligned = kp.Z_target_


# MMD is reported for source vs test only
print(f"MMD^2 (Zs vs Zt test, without Procrustes ): {mmd2_rbf(Zs, Zt_unaligned):.6f}")
print(f"MMD^2 (Zs vs Zt test, with Procrustes ):   {mmd2_rbf(Zs, Zt):.6f}")


In [ ]:
## Images before adaptation: Procrustes-aligned Zs_a / Za (shared pixel frame); test samples reuse the adapt-set gene coordinates
from kpcada import ImageTransformerDI

tf = ImageTransformerDI(discretization='bin', pixels=(224, 224)).fit(Zs_a, Za, plot=True)

# same scalers as the adapted images below: source scaler on X1_vf, target scaler fitted on the adaptation set only
mms_s = MinMaxScaler().fit(X1_vf)
mms_t = MinMaxScaler().fit(X2_adapt_vf)

images_source_pre = tf.transform(mms_s.transform(X1_vf), domain='source')        # (n_source_samples, 224, 224, 3)
images_adapt_pre  = tf.transform(mms_t.transform(X2_adapt_vf), domain='target')  # (n_adapt_samples, 224, 224, 3)
images_target_pre = tf.transform(mms_t.transform(X2_test_vf), domain='target')   # (n_target_test_samples, 224, 224, 3)
print(images_source_pre.shape, images_adapt_pre.shape, images_target_pre.shape)

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(images_source_pre[0]); ax[0].set_title('source sample 0')
ax[1].imshow(images_target_pre[0]); ax[1].set_title('target test sample 0')
plt.show()

## ResNet-50 on the pre-adaptation images
Images come from the source embedding and the Procrustes-aligned adapt-set embedding (`Zs_a`, `Za`) **before** the MMD step; test samples are mapped through the adapt-set gene coordinates. The classifier is trained on source images only; the best epoch is chosen on a source validation split and the target test set is only scored.

In [ ]:
from kpcada import train_eval_resnet50

In [ ]:
out_pre = train_eval_resnet50(
    images_source_pre, Y1,
    {"target_test": (images_target_pre, Y2_test)},
    epochs=15, batch_size=16, lr=1e-4, pretrained=True, seed=0, verbose=False,
)
print(f"Before adaptation - target test accuracy: {out_pre['results']['target_test']['accuracy']:.3f}")

### Adaptation on the Target using the Source

## KPCA-DA (updated `kpcada.py`, JAMIA methodology: MMD² + λ·L_KSP, Eqs. 5-9)
Source = all of `X1`; target = the 70% adaptation set `X2_adapt` only, both variance-filtered in cell 6 (`X1_vf`, `X2_adapt_vf`). The held-out `X2_test` is never used for the gene embedding, gamma or lambda.
Inputs are genes x samples, so the matrices are transposed.

In [ ]:
# adaptation code: kpcada/kpcada.py, imported through the kpcada package (put on sys.path in cell 1)
from kpcada import kpcada
KPCA_DA = kpcada.KPCA_DA
print("adaptation code:", kpcada.__file__)

Xs_g, Xa_g = X1_vf.T, X2_adapt_vf.T      # genes x samples (variance-filtered)

# Start from the Procrustes-aligned embeddings of the baseline cell: Zs (source, reference) and Za (adapt set rotated onto it).
# X is only used for K~_t (centered target kernel) in the KSP loss; kPCA and Procrustes are not repeated.
# label-free lambda sweep with an MMD^2 budget (never tuned on test accuracy): lambda = 0 gives the MMD^2 floor, then
# the largest lambda with MMD^2 <= floor + eps * (MMD^2_init - floor) is chosen (eps = 0.10); gamma from the median heuristic
# lr is scaled up because the MMD gradient is mean-normalised; K~_t uses the target kPCA bandwidth (default reg_gamma)
# stop when the total loss changes < 1e-6 (relative) over 20 epochs, or at max_epochs; lr is halved when a step
# raises the loss. Same settings for every lambda; lr_used / epochs_used per lambda are in the table below.
da = KPCA_DA(n_components=2, lr=200, max_epochs=1000, tol=1e-6, window=20)
sweep = da.tune_lambda(Xs_g, Xa_g, genes1_vf, genes2_vf,
                       lambda_values=[0, 1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0], epsilon=0.10,
                       Z_source_init=Zs_a, Z_target_init=Za)
print("MMD gamma:", sweep["mmd_gamma"])
print(f"MMD^2 init {sweep['mmd_init']:.6f}  floor {sweep['mmd_floor']:.6f}  gain {sweep['gain']:.6f}")
sel = sweep["selected"]
print(f"selected lambda = {sel['lambda']}  (MMD^2 {sel['final_mmd']:.6f}, KSP {sel['final_reg']:.4f})")
print("sensitivity (epsilon -> lambda):", {e: r["lambda"] for e, r in sweep["sensitivity"].items()})
display(sweep["results_df"])


In [ ]:
# fit with the selected lambda (da.lambda_reg and da.mmd_gamma were set by tune_lambda)
# starting point = Procrustes-aligned (Zs_a, Za); da.Z_t_initial_ is Za, Zt_da is the adapted version
Zs_da, Zt_da = da.fit_transform(Xs_g, Xa_g, genes1_vf, genes2_vf, Z_source_init=Zs_a, Z_target_init=Za)
ci_s, ci_t = da.common_source_idx_, da.common_target_idx_

import torch
g = da.mmd_gamma
Zs_c = torch.tensor(Zs_da[ci_s])
mmd_before = kpcada.mmd_loss(Zs_c, torch.tensor(da.Z_t_initial_[ci_t]), g).item()
mmd_after  = kpcada.mmd_loss(Zs_c, torch.tensor(Zt_da[ci_t]), g).item()

print(f"MMD :  (shared genes, adapt set) before adaptation: {mmd_before:.6f}   after: {mmd_after:.6f}")
print(f"lambda {da.lambda_reg}: lr_used {da.lr_used_:.3g}, epochs_used {da.epochs_used_}, converged {da.converged_}")

In [ ]:
# Source vs TEST, before and after adaptation. The test samples have no embedding of their own here: they are mapped through the
# gene coordinates of the adapt set (before = Procrustes-aligned Za, after = KPCA-DA-adapted Zt_da), the same coordinates the
# test images use. Same shared genes and the same MMD bandwidth (g) for every line.
def _mmd(Zsrc, Ztgt):
    # Za / Zt are float64 (numpy), Zs_da / Zt_da are float32 (torch); cast both to float32 for cdist
    a = torch.as_tensor(Zsrc[ci_s], dtype=torch.float32)
    b = torch.as_tensor(Ztgt[ci_t], dtype=torch.float32)
    return kpcada.mmd_loss(a, b, g).item()

print(f"MMD^2 source vs test, before adaptation (Procrustes only, via adapt coords): {_mmd(Zs_da, Za):.6f}")
print(f"MMD^2 source vs test, after  adaptation (KPCA-DA, via adapt coords):        {_mmd(Zs_da, Zt_da):.6f}") 

In [ ]:
# images from the adapted embeddings (shared pixel frame); test samples reuse the gene coordinates
tf_da = ImageTransformerDI(discretization='bin', pixels=(224, 224)).fit(Zs_da, Zt_da)
mms_s = MinMaxScaler().fit(X1_vf)                    # source scaler (source genes)
mms_t = MinMaxScaler().fit(X2_adapt_vf)              # target scaler fitted on the adaptation set only
images_source_da = tf_da.transform(mms_s.transform(X1_vf), domain='source')
images_adapt_da  = tf_da.transform(mms_t.transform(X2_adapt_vf), domain='target')
images_test_da   = tf_da.transform(mms_t.transform(X2_test_vf), domain='target')
print(images_source_da.shape, images_adapt_da.shape, images_test_da.shape)

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(images_source_da[0]); ax[0].set_title('source sample 0')
ax[1].imshow(images_adapt_da[0]); ax[1].set_title('target test sample 0')
plt.show()

In [ ]:
out_da = train_eval_resnet50(
    images_source_da, Y1,
    {"target_test": (images_test_da, Y2_test)},
    epochs=15, batch_size=16, lr=1e-4, pretrained=True, seed=0, verbose=False,
)
print(f"After adaptation - target test accuracy: {out_da['results']['target_test']['accuracy']:.3f}")

In [ ]:
## ICA of source vs target-test images, before and after domain adaptation.
## ONE ICA is fitted on the SOURCE images of the reference (before-adaptation) layout (flatten, standardise, whitened
## PCA, FastICA) and every image set of both arms is projected with it, so before and after adaptation are compared in
## a single, consistent coordinate system. The target test images are only transformed, never used to fit it.
import ot
from sklearn.decomposition import PCA, FastICA

def fit_source_ica(img_src, n_components=2, pca_dim=50, seed=0):
    """Fit pixel standardisation, whitened PCA and FastICA on the source images. Returns (transform, source ICs)."""
    flat = img_src.reshape(len(img_src), -1).astype(np.float32)
    keep = flat.std(0) > 1e-6                      # drop pixels that are constant in the source (background)
    flat = flat[:, keep]
    mu, sd = flat.mean(0), flat.std(0)             # per-pixel z-score with source statistics
    pca = PCA(n_components=min(pca_dim, *flat.shape), whiten=True, random_state=seed)
    ica = FastICA(n_components=n_components, whiten="unit-variance", random_state=seed, max_iter=2000)
    ic_src = ica.fit_transform(pca.fit_transform((flat - mu) / sd))

    def transform(img):
        x = img.reshape(len(img), -1).astype(np.float32)[:, keep]
        return ica.transform(pca.transform((x - mu) / sd))
    return transform, ic_src

def ica_distances(ic_s, ic_t):
    """Wasserstein-1 (POT exact EMD, Euclidean ground cost, uniform weights) and centroid distance in 2-D ICA space."""
    ic_s, ic_t = np.asarray(ic_s, dtype=np.float64), np.asarray(ic_t, dtype=np.float64)
    M = ot.dist(ic_s, ic_t, metric="euclidean")
    w1 = ot.emd2(ot.unif(len(ic_s)), ot.unif(len(ic_t)), M)
    return float(w1), float(np.linalg.norm(ic_s.mean(0) - ic_t.mean(0)))

def common_limits(pts, lo=0.5, hi=99.5, pad=0.2):
    out = []
    for dim in (0, 1):
        a, b = np.percentile(pts[:, dim], [lo, hi])
        m = (b - a) * pad
        out.append((a - m, b + m))
    return out

# The adaptation moves only the target genes, so the source images should be (nearly) the same in both arms.
print(f"max |source image before - after adaptation|: {np.abs(images_source_pre - images_source_da).max():.3g}")

# the single source-fitted ICA shared by both arms
ica_transform, _ = fit_source_ica(images_source_pre)

arms = [
    ("Before adaptation", images_source_pre, images_target_pre),
    ("After adaptation",  images_source_da,  images_test_da),
]

# One row per arm per repeat. Set REPEAT before this cell when looping the pipeline over repeats; rows of earlier
# repeats are kept, and re-running the cell for the same repeat replaces its rows.
REPEAT = globals().get("REPEAT", 0)
ica_results = [r for r in globals().get("ica_results", []) if r["repeat"] != REPEAT]

ics = []
for arm, img_s, img_t in arms:
    ic_s, ic_t = ica_transform(img_s), ica_transform(img_t)
    w1, cdist_ = ica_distances(ic_s, ic_t)
    ica_results.append({"repeat": REPEAT, "arm": arm, "W1": w1, "centroid_dist": cdist_,
                        "n_source": len(ic_s), "n_target_test": len(ic_t)})
    ics.append((ic_s, ic_t))

ica_df = pd.DataFrame(ica_results)
display(ica_df[ica_df["repeat"] == REPEAT])
if ica_df["repeat"].nunique() > 1:
    display(ica_df.groupby("arm", sort=False)[["W1", "centroid_dist"]].agg(["mean", "std"]))

# Plot: both panels are in the same ICA coordinates, so no per-arm re-mapping is needed.
# same random subset of source samples in both panels, as many as there are test samples
rng = np.random.default_rng(0)
n_src, n_tgt = len(ics[0][0]), len(ics[0][1])
n_plot = min(n_src, n_tgt)
src_idx = rng.choice(n_src, n_plot, replace=False)
tgt_idx = rng.choice(n_tgt, n_plot, replace=False)

# one shared pair of axis limits for both panels
xlim, ylim = common_limits(np.vstack([ic[idx] for ic_s, ic_t in ics for ic, idx in ((ic_s, src_idx), (ic_t, tgt_idx))]))

rows = ica_df[ica_df["repeat"] == REPEAT].set_index("arm")
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (arm, _, _), (ic_s, ic_t) in zip(axes, arms, ics):
    ax.scatter(ic_s[src_idx, 0], ic_s[src_idx, 1], s=40, alpha=0.6, c="tab:blue", edgecolor="white", linewidth=0.4,
               label=f"source (n={n_plot} of {n_src})", zorder=2)
    ax.scatter(ic_t[tgt_idx, 0], ic_t[tgt_idx, 1], s=14, alpha=0.95, c="tab:orange", edgecolor="none",
               label=f"target test (n={n_plot})", zorder=3)
    ax.set_xlim(xlim); ax.set_ylim(ylim)
    ax.set_title(f"{arm}\nW1 = {rows.loc[arm, 'W1']:.3f}, centroid distance = {rows.loc[arm, 'centroid_dist']:.3f}")
    ax.set_xlabel("IC 1")
    ax.set_ylabel("IC 2")
    ax.legend(frameon=False)
plt.tight_layout()
plt.show()

## Grad-CAM before and after adaptation
Following the paper's interpretability analysis: Grad-CAM (last ResNet-50 block, true-class target) is computed over the target test samples of each subtype, each pixel is mapped back to its gene through the fixed DeepInsight layout, and genes are selected in two stages (activation threshold tau = 0.5, then the 10 genes with the highest mean activation per subtype). The same procedure is run on the source images as the reference, on the adapted pipeline (`out_da`, `images_test_da`) and on the unadapted one (`out_pre`, `images_target_pre`), and the selected genes are compared with the source-reference genes and the PAM50 signature.

In [ ]:
## Grad-CAM helpers (JAMIA methodology): per-class mean Grad-CAM over target samples, threshold tau, top-k genes per class
import copy
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
from kpcada.resnet_baseline import _to_tensor

TAU, TOP_K = 0.5, 10
pam50_genes = set(np.loadtxt(data_root / "PAM50_gene_list.txt", dtype=str))

def class_gradcams(model, images, y_enc, batch_size=16):
    """Grad-CAM on the last ResNet block, targeting each sample's TRUE class; returns {class: mean CAM (H, W)}.
    Runs on a CPU copy of the trained model (the one returned by train_eval_resnet50 is on MPS/CUDA)."""
    net = copy.deepcopy(model).cpu().eval()
    x = _to_tensor(images)
    cams = []
    with GradCAM(model=net, target_layers=[net.layer4[-1]]) as cam:
        for i in range(0, len(x), batch_size):
            targets = [ClassifierOutputTarget(int(c)) for c in y_enc[i:i + batch_size]]
            cams.append(cam(input_tensor=x[i:i + batch_size], targets=targets))
    cams = np.concatenate(cams)
    return {c: cams[y_enc == c].mean(0) for c in np.unique(y_enc)}

def top_genes(cam, coords, gene_names, tau=TAU, k=TOP_K):
    """Pixel -> gene through the fixed DeepInsight layout (coords[i] = (row, col) of gene i).
    Stage 1: keep genes whose pixel activation >= tau. Stage 2: the k with the highest activation."""
    act = cam[coords[:, 0], coords[:, 1]]
    keep = np.where(act >= tau)[0]
    keep = keep[np.argsort(-act[keep], kind="stable")]
    return list(gene_names[keep[:k]]), int(len(keep))

def gradcam_report(model, images, y, coords, gene_names, classes):
    y_enc = np.searchsorted(classes, y)
    cams = class_gradcams(model, images, y_enc)
    mean_img = {c: images[y_enc == c].mean(0) for c in cams}
    genes = {c: top_genes(cams[c], coords, gene_names) for c in cams}
    return {"cams": cams, "mean_img": mean_img, "genes": genes}

def plot_gradcam_rows(reports, classes, tau=TAU):
    """One row per condition, one column per class: class-mean image with the thresholded Grad-CAM overlaid."""
    fig, axs = plt.subplots(len(reports), len(classes), figsize=(3.2 * len(classes), 3.2 * len(reports)),
                            constrained_layout=True, squeeze=False)
    for r, (name, rep) in enumerate(reports.items()):
        for c in range(len(classes)):
            ax = axs[r, c]; ax.axis("off")
            if c not in rep["cams"]:
                continue
            cam = rep["cams"][c].copy(); cam[cam < tau] = 0
            img = np.clip(rep["mean_img"][c] / max(rep["mean_img"][c].max(), 1e-8), 0, 1)
            ax.imshow(show_cam_on_image(img.astype(np.float32), cam.astype(np.float32), use_rgb=True))
            ax.text(0, 0, classes[c], c="white", ha="left", va="top", weight="bold", size="large")
            ax.text(img.shape[1] - 1, img.shape[0] - 1, f"{rep['genes'][c][1]} genes >= {tau}", c="white",
                    ha="right", va="bottom", weight="bold")
            if c == 0:
                ax.set_title(name, loc="left", fontsize=11)
    return fig


In [ ]:
## Grad-CAM before vs after adaptation. Target test images, true-class targets; source reference = the adapted pipeline's
## model on source images. Each pixel is mapped back to its gene through that condition's layout.
classes = out_da["classes"]
assert list(out_pre["classes"]) == list(classes)

reports = {
    "Source reference":            gradcam_report(out_da["model"],  images_source_da,  Y1,      tf_da.coords_for("source"), genes1_vf, classes),
    "KPCA-DA (after adaptation)":  gradcam_report(out_da["model"],  images_test_da,    Y2_test, tf_da.coords_for("target"), genes2_vf, classes),
    "No adaptation (before)":      gradcam_report(out_pre["model"], images_target_pre, Y2_test, tf.coords_for("target"),    genes2_vf, classes),
}
fig = plot_gradcam_rows(reports, classes)
plt.show()


In [ ]:
## Top genes per subtype, overlap with the source-reference genes and with the PAM50 signature
src_top = {c: set(g) for c, (g, _) in reports["Source reference"]["genes"].items()}
rows = []
for cond, rep in reports.items():
    for c, (g, n_pass) in rep["genes"].items():
        rows.append({"condition": cond, "class": classes[c], f"n genes >= {TAU}": n_pass,
                     "PAM50 hits": len(set(g) & pam50_genes),
                     "overlap with source top genes": len(set(g) & src_top.get(c, set())),
                     "top genes": ", ".join(g)})
gradcam_table = pd.DataFrame(rows)
with pd.option_context("display.max_colwidth", None, "display.width", 250):
    display(gradcam_table)

summary = gradcam_table.groupby("condition", sort=False)[["PAM50 hits", "overlap with source top genes"]].sum()
summary["selected genes"] = gradcam_table.groupby("condition", sort=False)["top genes"].apply(
    lambda s: sum(len(x.split(", ")) if x else 0 for x in s))
display(summary)
